In [1]:
# ===== 1. SETUP & CONNECTION =====
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.primitives import BackendSamplerV2
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
import numpy as np


# Load credentials and connect to backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=6)
print(f"Using backend: {backend.name}")


# Create samplers for both real hardware and simulator
sampler = Sampler(mode=backend)


noise_model = NoiseModel.from_backend(backend)
backend_sim = AerSimulator(noise_model=noise_model)
sampler_sim = BackendSamplerV2(backend=backend_sim)




# ===== 2. ORACLE FUNCTION =====
def dj_function(num_qubits):
    """Creates a random constant or balanced function oracle"""
    qc_dj = QuantumCircuit(num_qubits + 1)
   
    # Randomly flip ancilla (changes constant 0->1)
    if np.random.randint(0, 2):
        qc_dj.x(num_qubits)
   
    # 50% chance: return constant function
    if np.random.randint(0, 2):
        return qc_dj
   
    # Otherwise: make balanced function
    on_states = np.random.choice(
        range(2**num_qubits),
        2**num_qubits // 2,
        replace=False,
    )


    def add_cx(qc_dj, bit_string):
        for qubit, bit in enumerate(reversed(bit_string)):
            if bit == "1":
                qc_dj.x(qubit)
        return qc_dj
   
    for state in on_states:
        qc_dj = add_cx(qc_dj, f"{state:0{num_qubits}b}")
        qc_dj.mcx(list(range(num_qubits)), num_qubits)
        qc_dj = add_cx(qc_dj, f"{state:0{num_qubits}b}")


    return qc_dj




# ===== 3. BUILD DEUTSCH-JOZSA CIRCUIT =====
n = 5  # Number of input qubits


# Generate oracle
oracle = dj_function(n)
print(f"Oracle created with {oracle.num_qubits} qubits")


# Build the complete Deutsch-Jozsa circuit
qc_dj = QuantumCircuit(n + 1, n)
qc_dj.x(n)                    # Initialize ancilla to |1⟩
qc_dj.h(range(n + 1))         # Apply Hadamards (superposition)
qc_dj.barrier()
qc_dj.compose(oracle, inplace=True)  # Apply oracle
qc_dj.barrier()
qc_dj.h(range(n))             # Final Hadamards (interference)
qc_dj.measure(range(n), range(n))  # Measure


# Optional: visualize the circuit
qc_dj.draw("mpl")




# ===== 4. TRANSPILE FOR HARDWARE =====
target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level=3)
qc_isa = pm.run(qc_dj)


print(f"Original circuit: {qc_dj.depth()} depth, {sum(qc_dj.count_ops().values())} gates")
print(f"Transpiled circuit: {qc_isa.depth()} depth, {sum(qc_isa.count_ops().values())} gates")




# ===== 5. RUN AND INTERPRET =====
# Choose which to run:
USE_SIMULATOR = True  # Set to False for real hardware


if USE_SIMULATOR:
    print("\nRunning on simulator...")
    job = sampler_sim.run([qc_isa], shots=1)
else:
    print("\nSubmitting to real quantum hardware...")
    print("(This may take a while in queue)")
    job = sampler.run([qc_isa], shots=1)


# Get results WITH monitoring
print(f"\nJob ID: {job.job_id()}")
print("Waiting for job to complete...")


import time
while True:
    status = job.status()
    print(f"Status: {status}")  # ← Just print status directly
   
    # Check if done (works for both string and object)
    status_str = str(status)
    if 'DONE' in status_str or 'ERROR' in status_str or 'CANCELLED' in status_str:
        break
   
    time.sleep(10)  # Check every 10 seconds


print("\n✓ Job completed!")
res = job.result()
counts = res[0].data.c.get_counts()


print(f"\nMeasurement result: {counts}")


# Interpret
if "0" * n in counts:
    print("✓ Result: CONSTANT function")
else:
    print("✓ Result: BALANCED function")



Using backend: ibm_fez
Oracle created with 6 qubits
Original circuit: 4 depth, 19 gates
Transpiled circuit: 7 depth, 40 gates

Running on simulator...

Job ID: ed128622-ef3c-4637-9a73-100421cbf125
Waiting for job to complete...
Status: JobStatus.RUNNING
Status: JobStatus.DONE

✓ Job completed!

Measurement result: {'00000': 1}
✓ Result: CONSTANT function
